# Bốn khoảng giá và điểm review rượu — Khánh

**Mục tiêu:** Điểm review trung vị thay đổi thế nào giữa bốn khoảng giá?

**Trạng thái:** Phân tích khám phá sơ bộ trên các bản ghi có giá quan sát được, phục vụ buổi đánh giá tiến trình thứ Tư tuần 6. Bảng được tạo trên toàn mẫu có giá; split seed 42 đã chốt ngày 21/10/2026, chưa chạy lại bảng riêng trên train. Các kết quả dưới đây là thống kê mô tả; không phải kết quả đánh giá khả năng dự đoán trên dữ liệu chưa thấy.

**Dữ liệu:** `data/processed/wine_cleaned_final.csv` — Wine Reviews 130k đã qua pipeline cleaning của nhóm. Mỗi dòng được đếm là **một bản ghi review**, không mặc định là một chai duy nhất vì cùng `title` có thể còn nhiều review.

Notebook đi từ chọn mẫu → tính tứ phân vị → lập bảng → nhận xét. Chọn **Restart Kernel and Run All Cells** để chạy lại toàn bộ khi mở bằng Jupyter.

## 1. Đọc dữ liệu sạch

Chỉ đọc file sạch đã có. Đoạn xác định đường dẫn hỗ trợ mở notebook từ thư mục gốc dự án, `notebooks`, hoặc `notebooks/w5_progress_parts`.

Hai cột dùng trong phân tích là `price` (giá, đơn vị USD) và `points` (điểm review). Ta đếm mẫu có giá và thiếu giá trước khi chọn mẫu để người đọc biết bảng sau bao phủ phần nào của dữ liệu.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "w5_progress_parts":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "wine_cleaned_final.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Không tìm thấy data/processed/wine_cleaned_final.csv. "
        "Hãy mở notebook từ thư mục gốc dự án, notebooks hoặc w5_progress_parts "
        "và kiểm tra file dữ liệu sạch đã có."
    )

df = pd.read_csv(DATA_PATH)
missing_columns = {"price", "points"} - set(df.columns)
if missing_columns:
    raise ValueError(f"Dữ liệu thiếu cột bắt buộc: {sorted(missing_columns)}")
if df.empty:
    raise ValueError("Dữ liệu sạch không có bản ghi nào để phân tích.")

# Định dạng số theo cách viết tiếng Việt: 1.000,50.
def format_number(value, decimals=0):
    return f"{value:,.{decimals}f}".translate(str.maketrans({",": ".", ".": ","}))

def format_price(value):
    return format_number(value, 2).rstrip("0").rstrip(",")

n_total = len(df)
n_with_price = int(df["price"].notna().sum())
n_missing_price = int(df["price"].isna().sum())
print(f"Tổng số bản ghi sạch: {format_number(n_total)}")
print(f"Có giá quan sát: {format_number(n_with_price)}")
print(f"Thiếu giá: {format_number(n_missing_price)}")

Tổng số bản ghi sạch: 119.988
Có giá quan sát: 111.593
Thiếu giá: 8.395


## 2. Chọn và kiểm tra mẫu phân tích

Giữ những dòng có `price`; giá thiếu vẫn tồn tại trong file nguồn. Không ước tính giá để đưa vào bảng và không loại chai giá cao.

Các giá quan sát phải là số dương, hữu hạn; mọi bản ghi được chọn phải có điểm review hữu hạn. Nếu kiểm tra không đạt, notebook dừng để kiểm tra lại dữ liệu thay vì âm thầm bỏ thêm dòng. `.copy()` giúp các cột phân nhóm chỉ được thêm vào mẫu phân tích trong bộ nhớ.

In [2]:
sample = df.loc[df["price"].notna(), ["price", "points"]].copy()
if sample.empty:
    raise ValueError("Không có bản ghi có giá để chia bốn khoảng.")

for column in ["price", "points"]:
    try:
        sample[column] = pd.to_numeric(sample[column], errors="raise")
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Cột {column} phải chứa giá trị số hợp lệ.") from exc

if not (np.isfinite(sample["price"]) & sample["price"].gt(0)).all():
    raise ValueError("Giá quan sát phải dương và hữu hạn; hãy kiểm tra dữ liệu sạch.")
if not np.isfinite(sample["points"]).all():
    raise ValueError("Mẫu có giá chứa points bị thiếu hoặc không hữu hạn.")

assert len(sample) == n_with_price
assert sample.index.is_unique
print(f"Mẫu phân tích: {format_number(len(sample))} bản ghi.")
print(f"Tỷ lệ trên toàn bộ dữ liệu sạch: {format_number(len(sample) / n_total * 100, 2)}%.")
print("Kiểm tra đạt: giá dương, hữu hạn; điểm review đầy đủ và hữu hạn.")

Mẫu phân tích: 111.593 bản ghi.
Tỷ lệ trên toàn bộ dữ liệu sạch: 93,00%.
Kiểm tra đạt: giá dương, hữu hạn; điểm review đầy đủ và hữu hạn.


## 3. Tính tứ phân vị và chia bốn khoảng giá

Tứ phân vị là các mốc ở 25%, 50% và 75% trong phân bố giá đã sắp xếp. Chúng giúp tạo bốn khoảng dựa trên dữ liệu, thay vì tự đặt ngưỡng giá tùy ý. Trung vị là giá trị ở giữa khi sắp xếp, nên ít bị kéo bởi một số chai rất đắt hơn số trung bình.

`pd.qcut(..., q=4)` tính các mốc và gán nhóm Q1–Q4 theo giá tăng dần. Nhóm đầu gồm cả giá nhỏ nhất và mốc 25%; các nhóm sau **lớn hơn cận dưới, nhỏ hơn hoặc bằng cận trên**. Ví dụ, bản ghi đúng tại mốc 25% thuộc Q1.

Nhiều bản ghi có cùng giá nên số lượng mỗi nhóm có thể lệch khỏi 25%. Ta giữ các giá bằng nhau trong cùng nhóm, không tách chúng theo thứ tự dòng để ép nhóm bằng nhau. Nếu các mốc trùng nhau khiến không thể tạo đủ bốn khoảng, notebook báo lỗi rõ ràng.

In [3]:
tier_names = ["Q1", "Q2", "Q3", "Q4"]
try:
    sample["price_tier"], price_edges = pd.qcut(
        sample["price"], q=4, labels=tier_names, retbins=True, duplicates="raise"
    )
except ValueError as exc:
    raise ValueError(
        "Không thể tạo bốn khoảng giá phân biệt: các mốc tứ phân vị có thể trùng nhau. "
        "Cần kiểm tra phân bố giá, không tự giảm số nhóm hoặc tách các bản ghi đồng giá."
    ) from exc

assert len(price_edges) == 5 and np.all(np.diff(price_edges) > 0)
assert sample["price_tier"].notna().all()
assert sample["price_tier"].nunique() == 4
assert sample.groupby("price", observed=True)["price_tier"].nunique().eq(1).all()

print(f"Giá nhỏ nhất: {format_price(price_edges[0])} USD")
for percentile, edge in zip([25, 50, 75], price_edges[1:4]):
    print(f"Mốc {percentile}%: {format_price(edge)} USD")
print(f"Giá lớn nhất: {format_price(price_edges[-1])} USD (được giữ trong mẫu)")

Giá nhỏ nhất: 4 USD
Mốc 25%: 17 USD
Mốc 50%: 25 USD
Mốc 75%: 42 USD
Giá lớn nhất: 2.500 USD (được giữ trong mẫu)


## 4. Bảng số lượng, giá trung vị và điểm trung vị

Mỗi hàng là một khoảng giá. **Tỷ lệ mẫu** dùng mẫu có giá làm mẫu số. **Điểm trung vị** mô tả mức điểm ở giữa của từng nhóm; đây không phải điểm của mọi bản ghi trong nhóm.

Bảng được tính lại bằng `groupby` mỗi lần chạy. Các kiểm tra bên dưới bảo đảm tổng số dòng của bốn nhóm bằng mẫu có giá và không có dòng bị bỏ ngoài khoảng. Tổng tỷ lệ hiển thị có thể lệch rất nhỏ khỏi 100% do làm tròn.

In [4]:
summary = (
    sample.groupby("price_tier", observed=True)
    .agg(
        n_records=("points", "size"),
        median_price=("price", "median"),
        median_points=("points", "median"),
    )
    .reindex(tier_names)
)
summary["sample_pct"] = summary["n_records"] / len(sample) * 100

assert len(summary) == 4 and summary["n_records"].gt(0).all()
assert int(summary["n_records"].sum()) == len(sample)
assert np.isclose(summary["sample_pct"].sum(), 100.0)
for i, tier in enumerate(tier_names):
    prices = sample.loc[sample["price_tier"].eq(tier), "price"]
    lower_ok = prices.ge(price_edges[i]) if i == 0 else prices.gt(price_edges[i])
    assert (lower_ok & prices.le(price_edges[i + 1])).all()

price_ranges = [
    f"{format_price(price_edges[0])}–{format_price(price_edges[1])}"
] + [
    f"Trên {format_price(price_edges[i])}–{format_price(price_edges[i + 1])}"
    for i in range(1, 4)
]

# Tách bảng hiển thị khỏi bảng số để giữ độ chính xác cho phần nhận xét.
summary_display = pd.DataFrame({
    "Nhóm": tier_names,
    "Khoảng giá (USD)": price_ranges,
    "Số bản ghi": summary["n_records"].map(format_number).to_numpy(),
    "Tỷ lệ mẫu (%)": summary["sample_pct"].map(lambda x: format_number(x, 2)).to_numpy(),
    "Giá trung vị (USD)": summary["median_price"].map(format_price).to_numpy(),
    "Điểm trung vị": summary["median_points"].map(lambda x: format_number(x, 1)).to_numpy(),
})
summary_display

,Nhóm,Khoảng giá (USD),Số bản ghi,Tỷ lệ mẫu (%),Giá trung vị (USD),Điểm trung vị
0,Q1,4–17,29.326,"26,28",13,"86,0"
1,Q2,Trên 17–25,27.105,"24,29",21,"88,0"
2,Q3,Trên 25–42,27.489,"24,63",33,"89,0"
3,Q4,Trên 42–2.500,27.673,"24,80",60,"91,0"


## 5. Ba nhận xét từ kết quả thực tế

Cell dưới lấy trực tiếp các số trong bảng, vì vậy khi dữ liệu thay đổi thì nhận xét cũng được cập nhật. Chênh lệch điểm được báo cáo bằng **điểm**, không gọi là phần trăm chất lượng. So sánh giá trung vị là tỷ số giữa hai mức giá đại diện, không đo mức cải thiện chất lượng.

In [5]:
medians = summary["median_points"]
median_text = "; ".join(format_number(value, 1) for value in medians)
if (np.diff(medians.to_numpy()) > 0).all():
    trend = "tăng qua cả bốn khoảng giá"
elif (np.diff(medians.to_numpy()) >= 0).all():
    trend = "không giảm qua bốn khoảng giá, nhưng có nhóm bằng nhau"
else:
    trend = "không tăng đều qua cả bốn khoảng giá"
print(f"1. Điểm trung vị từ Q1 đến Q4 lần lượt là {median_text}; {trend}.")

first, last = summary.iloc[0], summary.iloc[-1]
point_gap = last["median_points"] - first["median_points"]
price_ratio = last["median_price"] / first["median_price"]
print(
    f"2. Giá trung vị Q1 là {format_price(first['median_price'])} USD, "
    f"Q4 là {format_price(last['median_price'])} USD "
    f"(Q4 gấp {format_number(price_ratio, 2)} lần Q1). "
    f"Chênh lệch điểm trung vị Q4 − Q1 là {format_number(point_gap, 1)} điểm. "
    "Các số này mô tả sự khác biệt giữa nhóm, không phải tác động của việc tăng giá."
)
print(
    f"3. Mỗi nhóm chiếm từ {format_number(summary['sample_pct'].min(), 2)}% "
    f"đến {format_number(summary['sample_pct'].max(), 2)}% mẫu có giá. "
    "Số bản ghi không bằng nhau vì các bản ghi đồng giá được giữ cùng nhóm."
)

1. Điểm trung vị từ Q1 đến Q4 lần lượt là 86,0; 88,0; 89,0; 91,0; tăng qua cả bốn khoảng giá.
2. Giá trung vị Q1 là 13 USD, Q4 là 60 USD (Q4 gấp 4,62 lần Q1). Chênh lệch điểm trung vị Q4 − Q1 là 5,0 điểm. Các số này mô tả sự khác biệt giữa nhóm, không phải tác động của việc tăng giá.
3. Mỗi nhóm chiếm từ 24,29% đến 26,28% mẫu có giá. Số bản ghi không bằng nhau vì các bản ghi đồng giá được giữ cùng nhóm.


## 6. Giới hạn và cách trình bày

- **Phạm vi mẫu:** chỉ mô tả các bản ghi có giá. Nhóm thiếu giá có thể khác nhóm có giá; không mặc định kết luận áp dụng cho toàn bộ dữ liệu hay thị trường rượu.
- **Ý nghĩa của trung vị:** điểm trung vị nhóm cao hơn không bảo đảm từng chai trong nhóm đều có điểm cao hơn chai thuộc nhóm khác. Bảng này chưa mô tả đầy đủ phân bố hoặc độ chồng lấp của điểm.
- **Điểm review và quan hệ giá–điểm:** `points` phản ánh đánh giá của người chấm, không phải thước đo chất lượng khách quan. Quốc gia, giống nho, vùng và người đánh giá có thể liên quan đến cả giá lẫn điểm; bảng mô tả không chứng minh giá cao gây ra điểm cao.
- **Mức độ bằng chứng:** đây là khám phá sơ bộ trên mẫu có giá; không có kiểm định, khoảng tin cậy hay metric mô hình. Khi split được duyệt, kiểm định và mô hình phải tuân theo train/validation/test đã khóa.

**Gợi ý trình bày:** nói rõ nguồn và số mẫu; giải thích ba mốc tứ phân vị; đọc bảng và chênh lệch điểm trung vị Q4 − Q1; kết thúc bằng câu hỏi tiếp theo: “Thông tin sản phẩm có bổ sung khả năng dự đoán ngoài giá hay không?”